# Finance Advisor — RAG Pipeline Testing
Test PDF ingestion, embedding, Pinecone upsert, and retrieval end-to-end.

In [1]:
import sys
sys.path.insert(0, '..')

from dotenv import load_dotenv
load_dotenv('../.env')
print('Environment loaded')

Environment loaded


## 1. Generate Sample Bank Statement

In [2]:
import sys
sys.path.insert(0, '..')
from data.sample_statements.generate_sample import generate_bank_statement

pdf_path = generate_bank_statement('../data/sample_statements/sample_statement_rahul_sharma.pdf')
print(f'Generated: {pdf_path}')

ModuleNotFoundError: No module named 'reportlab'

## 2. Extract Text from PDF

In [ ]:
from src.rag.pdf_ingestion import extract_text_from_pdf

extraction = extract_text_from_pdf('../data/sample_statements/sample_statement_rahul_sharma.pdf')
print(f'Pages: {extraction["total_pages"]}')
print(f'Characters: {extraction["char_count"]}')
print('\nFirst 500 chars:')
print(extraction['full_text'][:500])

## 3. Chunk and Create Documents

In [ ]:
from src.rag.pdf_ingestion import create_documents

docs = create_documents(extraction, doc_type='bank_statement')
print(f'Total chunks: {len(docs)}')
print('\nSample chunk:')
print(docs[0])

## 4. Test Embeddings

In [ ]:
from src.rag.embeddings import embed_texts, embed_query, get_embedding_dimension

print(f'Embedding dimension: {get_embedding_dimension()}')

test_texts = ['SIP investment in mutual fund', 'salary credit from employer']
embeddings = embed_texts(test_texts)
print(f'Embedded {len(embeddings)} texts, dimension: {len(embeddings[0])}')

query_emb = embed_query('What are my monthly SIP investments?')
print(f'Query embedding dimension: {len(query_emb)}')

## 5. Upsert to Pinecone

In [ ]:
from src.rag.retriever import upsert_documents, clear_namespace

# Clear existing data
clear_namespace('bank_statements')

# Upsert bank statement chunks
n = upsert_documents(docs, namespace='bank_statements')
print(f'Upserted {n} documents to Pinecone')

## 6. Ingest SEBI Regulations

In [ ]:
from src.rag.pdf_ingestion import ingest_text_file
from src.rag.retriever import upsert_documents

reg_docs = ingest_text_file('../data/regulations/sebi_regulations.txt', doc_type='regulation')
print(f'Regulation chunks: {len(reg_docs)}')

upsert_documents(reg_docs, namespace='regulations')
print('Regulations ingested!')

## 7. Test Retrieval

In [ ]:
from src.rag.retriever import hybrid_retrieve, format_context

# Test bank statement retrieval
query = 'What are my top 3 expense categories?'
bank_docs = hybrid_retrieve(query, top_k=5, namespace='bank_statements')
print(f'Retrieved {len(bank_docs)} bank docs')

# Test regulation retrieval
reg_query = 'What is the maximum 80C deduction?'
reg_docs_retrieved = hybrid_retrieve(reg_query, top_k=3, namespace='regulations')
print(f'Retrieved {len(reg_docs_retrieved)} regulation docs')

print('\nContext:')
print(format_context(reg_docs_retrieved[:2]))

## 8. Test Financial Calculators

In [ ]:
from src.tools.financial_calculators import (
    calculate_sip, calculate_ppf, calculate_80c_tax_saving, adjust_for_inflation
)

print('=== SIP Calculator ===')
print(calculate_sip.invoke({'monthly_investment': 5000, 'annual_return_rate': 12, 'years': 20}))

print('\n=== PPF Calculator ===')
print(calculate_ppf.invoke({'annual_investment': 150000, 'years': 15}))

print('\n=== Tax Saving ===')
print(calculate_80c_tax_saving.invoke({
    'annual_income': 1020000,
    'investments_80c': 30000,
    'nps_contribution': 0,
    'regime': 'old'
}))

## 9. Full Pipeline Test (LangGraph)

In [ ]:
from src.graph.workflow import run_finance_advisor

result = run_finance_advisor(
    user_query='What are my top spending categories and how much can I save in tax this year?',
    user_profile={'age': 30, 'income': 85000, 'risk_appetite': 'moderate', 'investments_80c': 30000},
    stream=False,
)

print('=== SUMMARY ===')
print(result.get('summary', 'N/A'))
print('\n=== FINAL ADVICE (first 1000 chars) ===')
print((result.get('final_advice') or '')[:1000])